# 构建结构因果图

In [1]:
import os
import re
import pickle
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from time import time
from sklearn.preprocessing import StandardScaler
from IPython.lib.display import IFrame
from causalnex.structure import StructureModel
from causalnex.structure.notears import from_pandas, from_pandas_lasso
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE

## 1. Prepating the Data for Structure Learning

In [12]:
# 读取数据
with open('activity_space_areas_dict.pkl', 'rb') as f:
    act_space_data = pickle.load(f)

print(f"数据中共有 {len(act_space_data)} 个key")

数据中共有 110261 个key


In [13]:
# 将字典转换为DataFrame
data_list = []
for pid, (areas, clusters) in act_space_data.items():
    
    data_list.append({
        'pid': pid,
        'area0': areas[0] if len(areas) > 0 else None,
        'area1': areas[1] if len(areas) > 1 else None,
        'area2': areas[2] if len(areas) > 2 else None,
        'area3': areas[3] if len(areas) > 3 else None,
        'cluster0': clusters[0] if len(clusters) > 0 else None,
        'cluster1': clusters[1] if len(clusters) > 1 else None,
        'cluster2': clusters[2] if len(clusters) > 2 else None,
        'cluster3': clusters[3] if len(clusters) > 3 else None
    })

df = pd.DataFrame(data_list)

# 显示前几行查看结果
df.head()

,pid,area0,area1,area2,area3,cluster0,cluster1,cluster2,cluster3
0,270.0,3.147451,3.308764,3.235937,3.434450,1,1,1,1.0
1,2753.0,3.404565,3.226078,3.234076,3.220988,1,1,1,1.0
2,3203.0,3.352930,3.992503,3.329387,3.443081,2,2,2,2.0
3,4100.0,4.208079,4.887184,5.050174,4.030251,2,2,2,2.0
4,4658.0,3.530046,3.069087,3.043223,3.064848,2,1,1,1.0


In [14]:
# 计算每行的缺失值情况
print("\n各列缺失值统计:")
print(df.isnull().sum())



各列缺失值统计:
pid          0
area0        0
area1        0
area2        0
area3        0
cluster0     0
cluster1     0
cluster2     0
cluster3    29
dtype: int64


In [15]:
# 删除有缺失值的行
df = df.dropna()
print(f"\n删除缺失值后剩余样本量: {len(df)}")

# 将cluster3列转化为int类型
df['cluster3'] = df['cluster3'].astype(int)


删除缺失值后剩余样本量: 110232


In [16]:
# 导入变量信息
variables = pd.read_csv('../../autodl-fs/all_features_v1.csv')
print(variables.columns)

# 连续值建成环境变化
treat_data = pd.read_csv("../../autodl-fs/env_diff.csv")
print("\n建成环境变化量数据：")
print(treat_data.columns)

# # 静态协变量和分类处置变量 - covariance
# covariate = pd.read_csv('../../autodl-fs/df_cov_and_treat.csv')

# # 导入建成环境变量，用于补充静态协变量中的迁居后建成环境数据
# be_data = pd.read_csv('../../autodl-fs/relocate_information_gpd.csv')

Index(['pid', 'year', 'week', 'avg_daily_slow_dist', 'slow_dist_ratio',
       'home_year', 'home_week', 'work_year', 'work_week', 'time', 'home_time',
       'work_time', 'home_density', 'home_landusemix', 'home_road_density',
       'home_openratio_wide', 'home_subway', 'home_bus', 'home_center',
       'work_density', 'work_landusemix', 'work_road_density',
       'work_openratio_wide', 'work_subway', 'work_bus', 'work_center',
       'commute_distance', 'gender', 'age', 'edu', 'married', 'income', 'z_0',
       'z_1', 'home_density_label', 'home_landusemix_label',
       'home_road_density_label', 'home_openratio_wide_label',
       'home_subway_label', 'home_bus_label', 'home_center_label',
       'work_density_label', 'work_landusemix_label',
       'work_road_density_label', 'work_openratio_wide_label',
       'work_subway_label', 'work_bus_label', 'work_center_label',
       'home_change', 'work_change', 'state', 'weekly_R24H', 'weekly_AVGT',
       'is_workweek', 'previous_avg

In [17]:
print(variables['pid'].nunique())
print(treat_data['pid'].nunique())

138032
147440


In [18]:
# 计算df和variables的pid重合情况
df_pid_set = set(df['pid'])
variables_pid_set = set(variables['pid'])
treat_data_pid_set = set(treat_data['pid'])

print("df和variables的pid重合情况:")
print(f"df中的pid数量: {len(df_pid_set)}")
print(f"variables中的pid数量: {len(variables_pid_set)}")
print(f"重合的pid数量: {len(df_pid_set.intersection(variables_pid_set))}")

print("\ndf和treat_data的pid重合情况:")
print(f"df中的pid数量: {len(df_pid_set)}")
print(f"treat_data中的pid数量: {len(treat_data_pid_set)}")
print(f"重合的pid数量: {len(df_pid_set.intersection(treat_data_pid_set))}")


df和variables的pid重合情况:
df中的pid数量: 110232
variables中的pid数量: 138032
重合的pid数量: 84860

df和treat_data的pid重合情况:
df中的pid数量: 110232
treat_data中的pid数量: 147440
重合的pid数量: 84860


In [19]:
# 选取df中与variables和treat_data都重合的pid
common_pid = df_pid_set.intersection(variables_pid_set).intersection(treat_data_pid_set)
print(f"\n三个数据集共同的pid数量: {len(common_pid)}")

# 筛选出共同的pid
df = df[df['pid'].isin(common_pid)].reset_index(drop=True)
print(f"筛选后df的样本量: {len(df)}")


三个数据集共同的pid数量: 84860
筛选后df的样本量: 84860


用84,860样本也可以

### 选取需要的列

In [3]:
be_cols = ['home_density', 'home_landusemix', 'home_road_density',
       'home_openratio_wide', 'home_subway', 'home_bus', 'home_center',
       'work_density', 'work_landusemix', 'work_road_density',
       'work_openratio_wide', 'work_subway', 'work_bus', 'work_center',
       'commute_distance']

user_cols = ['gender', 'age', 'edu', 'married', 'income', 'state', 'z_0', 'z_1'] #加上两个混淆替代，看看效果？

In [21]:
# 按pid和time升序排序
variables = variables.sort_values(['pid', 'time'])

# 提取每个pid的第一行和最后一行的建成环境数据
be_0 = variables.groupby('pid')[be_cols].first().reset_index()
be_1 = variables.groupby('pid')[be_cols].last().reset_index()

# 重命名列,添加后缀
be_0.columns = ['pid'] + [col + '_0' for col in be_cols]
be_1.columns = ['pid'] + [col + '_1' for col in be_cols]

# 提取每个pid的第一行的用户特征数据
user_features = variables.groupby('pid')[user_cols].first().reset_index()

# 合并数据
features_df = pd.merge(be_0, be_1, on='pid')
features_df = pd.merge(features_df, user_features, on='pid')

features_df.head()

,pid,home_density_0,home_landusemix_0,home_road_density_0,home_openratio_wide_0,home_subway_0,home_bus_0,home_center_0,work_density_0,work_landusemix_0,...,work_center_1,commute_distance_1,gender,age,edu,married,income,state,z_0,z_1
0,263.0,19.180278,0.674795,28.261657,0.229073,0.000000,0.646465,4.266739,9.033984,0.785131,...,9.644711,1.828381,1,1,0.042208,0.364399,1,1,1.115773,-0.136993
1,270.0,14.529677,0.618902,38.814044,0.050432,0.000000,0.177778,3.473704,23.875621,0.577250,...,10.653325,0.190520,1,1,0.019883,0.497199,0,1,-0.318092,0.054852
2,2753.0,43.323924,0.731403,21.825231,0.021142,0.000000,10.666667,9.977737,25.321366,0.679151,...,9.977737,1.431244,2,2,0.064334,0.533578,0,1,-0.951096,0.132268
3,3605.0,72.557727,0.716023,34.220444,0.006849,0.000000,10.666667,0.726368,11.029024,0.599132,...,2.553382,5.859032,1,1,0.185533,0.485299,1,0,-1.581614,0.670394
4,4100.0,32.723039,0.688787,29.601778,0.041243,1.185185,11.851852,9.260486,0.024889,0.802058,...,6.693424,9.135008,1,1,0.084597,0.329650,1,1,-1.128722,0.139727


In [22]:
# 根据pid关联df和features_df
df = pd.merge(df, features_df, on='pid', how='left')
df.head()

,pid,area0,area1,area2,area3,cluster0,cluster1,cluster2,cluster3,home_density_0,...,work_center_1,commute_distance_1,gender,age,edu,married,income,state,z_0,z_1
0,270.0,3.147451,3.308764,3.235937,3.434450,1,1,1,1,14.529677,...,10.653325,0.190520,1,1,0.019883,0.497199,0,1,-0.318092,0.054852
1,2753.0,3.404565,3.226078,3.234076,3.220988,1,1,1,1,43.323924,...,9.977737,1.431244,2,2,0.064334,0.533578,0,1,-0.951096,0.132268
2,4100.0,4.208079,4.887184,5.050174,4.030251,2,2,2,2,32.723039,...,6.693424,9.135008,1,1,0.084597,0.329650,1,1,-1.128722,0.139727
3,4658.0,3.530046,3.069087,3.043223,3.064848,2,1,1,1,26.280335,...,11.444046,0.536638,1,1,0.283376,0.357079,2,1,-0.465670,0.074777
4,4942.0,3.458569,5.794304,3.526932,3.464373,2,2,1,2,21.517534,...,6.615958,3.437476,1,2,0.410029,0.596804,3,1,-0.229027,0.062306


In [23]:
# 计算包含nan的行数
nan_rows = df.isna().any(axis=1).sum()
print(f"包含nan值的行数: {nan_rows}")


包含nan值的行数: 0


In [24]:
# 从treat_data中选取建成环境变量的变化量
treat_data.rename(columns={col: col + '_change' for col in be_cols}, inplace=True)
df = pd.merge(df, treat_data, on='pid', how='left')

nan_rows = df.isna().any(axis=1).sum()
print(f"包含nan值的行数: {nan_rows}")

包含nan值的行数: 0


In [25]:
# 保存一下
df.to_csv('../data/act_space_features.csv', index=False)

## 2. Apply the NOTEARS algorithm to learn the structure

In [2]:
# 导入df
# df = pd.read_csv('../data/act_space_features.csv')
df = pd.read_csv('../data/act_space_travel_features.csv')

# 删除pid列
df = df.drop('pid', axis=1)

# 删除be_change列和两个潜在替代
treat_cols = ['home_density_change','home_landusemix_change','home_road_density_change','home_openratio_wide_change', 
              'home_subway_change','home_bus_change','home_center_change','work_density_change','work_landusemix_change',
              'work_road_density_change','work_openratio_wide_change','work_subway_change','work_bus_change','work_center_change']
# treat_cols = []
df = df.drop(columns=treat_cols+['z_0','z_1'])

参考Wang2019的论文，出行变量选择：
1. Total travel time
2. Slow trip frequency
3. Subway trip frequency

因此删除avg_daily_dist, avg_daily_tube_dist, avg_daily_slow_dist试试
* **删除之后结果不好！**

In [100]:
# 定义需要删除的列前缀
# drop_travel_cols = ['avg_daily_dur']

# # 使用 list comprehension 和 filter 来获取匹配的列
# cols_to_drop = [col for col in df.columns if any(col.startswith(prefix) for prefix in drop_travel_cols)]

# # 删除这些列
# df = df.drop(columns=cols_to_drop)

# print(df.columns)

In [3]:
# 对非int类型的列进行标准化
numeric_cols = df.select_dtypes(include=['float64']).columns
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

df.head()

,area0,area1,area2,area3,cluster0,cluster1,cluster2,cluster3,home_density_0,home_landusemix_0,...,avg_daily_slow_freq2,avg_daily_slow_freq3,avg_daily_tube_dist0,avg_daily_tube_dist1,avg_daily_tube_dist2,avg_daily_tube_dist3,avg_daily_tube_freq0,avg_daily_tube_freq1,avg_daily_tube_freq2,avg_daily_tube_freq3
0,-0.744687,-0.608889,-0.692689,-0.509501,1,1,1,1,-0.885528,-0.611027,...,-0.517820,-0.507870,-0.912620,-0.893552,-0.905257,-0.913905,-1.115959,-1.068603,-1.119792,-1.101909
1,-0.480117,-0.683268,-0.694503,-0.728679,1,1,1,1,0.256275,0.735790,...,-0.696064,-0.701271,0.024985,0.017771,-0.372836,-0.329942,-0.517611,-0.507939,-0.469989,-0.441959
2,0.346698,0.810954,1.074974,0.102256,2,2,2,2,-0.164091,0.225602,...,-0.606139,-0.406450,0.584033,0.056909,-0.031343,0.180311,0.512307,0.123323,0.305134,0.894306
3,-0.350997,-0.824487,-0.880456,-0.889001,2,1,1,1,-0.419569,-0.048852,...,-0.747303,-0.740707,0.222559,0.875306,0.274114,0.255988,0.146706,0.694621,-0.176184,-0.266391
4,-0.424547,1.626939,-0.409164,-0.478776,2,2,1,2,-0.608432,1.129840,...,-0.080472,-0.353129,-0.912620,-0.893552,-0.844739,-0.847529,-1.115959,-1.068603,-0.859871,-0.837929


### Area + Cluster

设定规则：
1. 调整后的活动空间不作为父节点（感兴趣的outcomes）
2. 一些个人属性（age, gender）不可作为子节点

In [4]:
t0 = time()

sm = from_pandas(df, tabu_parent_nodes=['area2','area3', 'cluster2','cluster3'],tabu_child_nodes=['age', 'gender'], w_threshold=0.02)
print(f'Running NOTEARS algorithm takes {time() - t0} seconds')

Running NOTEARS algorithm takes 156.24037098884583 seconds


In [5]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("supporting_files/00_initial_graph.html")

supporting_files/00_initial_graph.html


In [6]:
sm_subgraph = sm.get_largest_subgraph()
viz = plot_structure(
    sm_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("supporting_files/00_largest_subgraph.html")

supporting_files/00_largest_subgraph.html


In [4]:
# 把cluster和area分开求解
df_area = df.drop(columns = ['cluster0', 'cluster1', 'cluster2', 'cluster3'])
df_cluster = df.drop(columns = ['area0', 'area1', 'area2', 'area3'])

### Area Causal Path

In [22]:
sm_area = from_pandas(df_area)

In [5]:
# sm_area = from_pandas(df_area, tabu_parent_nodes=['area0','area1','area2','area3'])
sm_area = from_pandas(df_area, tabu_parent_nodes=['area2','area3'])

In [6]:
print(sm_area.adj)

{'area0': {}, 'area1': {}, 'area2': {}, 'area3': {}, 'home_density_0': {'area0': {'origin': 'learned', 'weight': 0.002701812793478628}, 'area1': {'origin': 'learned', 'weight': 0.0015345652850422826}, 'area2': {'origin': 'learned', 'weight': 0.0010476506660405999}, 'area3': {'origin': 'learned', 'weight': 0.005548982047078525}, 'home_landusemix_0': {'origin': 'learned', 'weight': -0.0004714085444269733}, 'home_road_density_0': {'origin': 'learned', 'weight': 0.008421103853292929}, 'home_openratio_wide_0': {'origin': 'learned', 'weight': -0.006208479938883276}, 'home_subway_0': {'origin': 'learned', 'weight': 0.0009644809660208216}, 'home_bus_0': {'origin': 'learned', 'weight': -0.000448214568630643}, 'home_center_0': {'origin': 'learned', 'weight': -0.006396041486483252}, 'work_density_0': {'origin': 'learned', 'weight': 0.00011399895529959281}, 'work_landusemix_0': {'origin': 'learned', 'weight': 0.0018811029860441083}, 'work_road_density_0': {'origin': 'learned', 'weight': 0.00547672

visualise the learned StructureModel using the plot function.

In [10]:
viz = plot_structure(
    sm_area,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("supporting_files/01_fully_connected.html")

supporting_files/01_fully_connected.html


#### **apply threshold**

In [19]:
weight_thre = 0.02

In [20]:
sm_area.remove_edges_below_threshold(weight_thre)

viz = plot_structure(
    sm_area,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
# viz.toggle_physics(False)
viz.show("supporting_files/01_thresholded.html")

supporting_files/01_thresholded.html


#### **最大子图**

In [21]:
sm_area_subgraph = sm_area.get_largest_subgraph()
viz = plot_structure(
    sm_area_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("supporting_files/01_largest_subgraph.html")

supporting_files/01_largest_subgraph.html


### **修改结构**

In [7]:
# 获取所有的边
edges1 = list(sm_subgraph.edges)

* 规则1：父节点的时期不应该晚于子节点

In [8]:
# 函数：提取节点名称中的数字部分
def get_time_index(node_name):
    match = re.search(r'(\d+)$', node_name)  # 匹配节点名中的数字部分
    return int(match.group(1)) if match else None

In [9]:
# 遍历所有边，删除不符合因果关系的边
edges_to_remove = []
for parent, child in edges1:
    parent_time = get_time_index(parent)
    child_time = get_time_index(child)
    
    if parent_time is not None and child_time is not None and parent_time > child_time:  # 如果父节点时间序号大于子节点时间序号，删除这条边
        edges_to_remove.append((parent, child))

In [10]:
# 从图中删除这些边
sm_subgraph.remove_edges_from(edges_to_remove)

viz = plot_structure(
    sm_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.show("supporting_files/00_modified_structure.html")

supporting_files/00_modified_structure.html


* 规则2：同一时期的出行变量之间的因果路径删除

In [11]:
edges2 = list(sm_subgraph.edges)

In [12]:
# 定义出行变量前缀
travel_cols = [
    'avg_daily_dist', 'avg_daily_dur', 'avg_daily_tube_dist', 'avg_daily_tube_freq',
    'avg_daily_slow_dist', 'avg_daily_slow_freq'
]

In [13]:
# 删除同一时期内出行变量之间的因果路径
edges_to_remove = []
for parent, child in edges2:
    # 检查父节点和子节点是否属于同一时期的变量
    if any(parent.startswith(prefix) for prefix in travel_cols) and any(child.startswith(prefix) for prefix in travel_cols):
        parent_time = get_time_index(parent)
        child_time = get_time_index(child)
        
        # 如果父节点和子节点属于同一时期，删除这条边
        if parent_time is not None and child_time is not None and parent_time == child_time:
            edges_to_remove.append((parent, child))

In [14]:
# 从图中删除这些边
sm_subgraph.remove_edges_from(edges_to_remove)

viz = plot_structure(
    sm_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.show("supporting_files/00_modified_structure.html")

supporting_files/00_modified_structure.html


In [30]:
list(sm_area_subgraph.edges)

[('area0', 'area2'),
 ('area0', 'area3'),
 ('area0', 'avg_daily_slow_dist0'),
 ('area0', 'avg_daily_slow_dist1'),
 ('area1', 'area2'),
 ('area1', 'area3'),
 ('area1', 'avg_daily_slow_dist1'),
 ('home_density_0', 'avg_daily_tube_freq0'),
 ('work_landusemix_0', 'work_road_density_0'),
 ('work_landusemix_0', 'work_bus_0'),
 ('work_openratio_wide_0', 'work_landusemix_0'),
 ('work_openratio_wide_0', 'work_road_density_0'),
 ('work_openratio_wide_0', 'work_bus_0'),
 ('work_subway_0', 'work_bus_0'),
 ('work_subway_0', 'avg_daily_tube_dist0'),
 ('work_subway_0', 'avg_daily_tube_freq0'),
 ('work_subway_0', 'avg_daily_tube_freq1'),
 ('work_subway_0', 'avg_daily_tube_freq2'),
 ('work_subway_0', 'avg_daily_tube_freq3'),
 ('commute_distance_0', 'area3'),
 ('commute_distance_0', 'avg_daily_dist0'),
 ('commute_distance_0', 'avg_daily_dist1'),
 ('commute_distance_0', 'avg_daily_dist2'),
 ('commute_distance_0', 'avg_daily_dist3'),
 ('commute_distance_0', 'avg_daily_dur0'),
 ('commute_distance_0', 'avg_

* 规则3：删除avg_daily_dist, avg_daily_tube_dist, avg_daily_slow_dist试试

In [15]:
# 获取所有边
edges3 = list(sm_subgraph.edges)

In [16]:
# 定义需要删除的列名
remove_columns = ['avg_daily_dist', 'avg_daily_tube_dist', 'avg_daily_slow_dist']

# 筛选出与 'avg_daily_dist', 'avg_daily_tube_dist', 'avg_daily_slow_dist' 相关的边
edges_to_remove = [(parent, child) for parent, child in edges3 if any(col in parent for col in remove_columns) or any(col in child for col in remove_columns)]

In [17]:
# 删除这些边
sm_subgraph.remove_edges_from(edges_to_remove)

# 重新获取最大子图
sm_subgraph = sm_subgraph.get_largest_subgraph()

# 可视化更新后的图形并保存为 HTML 文件
viz = plot_structure(
    sm_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.show("supporting_files/00_modified_structure.html")

supporting_files/00_modified_structure.html


* 规则4：添加边（area2->area3; cluster2->cluster3）

In [18]:
sm_subgraph.add_edge("area2", "area3")
sm_subgraph.add_edge("cluster2", "cluster3")

In [19]:
# 可视化更新后的图形并保存为 HTML 文件
viz = plot_structure(
    sm_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.show("supporting_files/00_modified_structure.html")

supporting_files/00_modified_structure.html


### 导出结构

In [47]:
import networkx as nx

nx.drawing.nx_pydot.write_dot(sm_subgraph, 'supporting_files/graph.dot')

### Cluster Causal Path

In [50]:
sm_cluster = from_pandas(df_cluster, tabu_parent_nodes=['cluster2','cluster3'])

In [51]:
weight_thre = 0.03

In [52]:
sm_cluster.remove_edges_below_threshold(weight_thre)

viz = plot_structure(
    sm_cluster,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
# viz.toggle_physics(False)
viz.show("supporting_files/02_thresholded.html")

supporting_files/02_thresholded.html


**最大子图**

In [53]:
sm_cluster_subgraph = sm_cluster.get_largest_subgraph()
viz = plot_structure(
    sm_cluster_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("supporting_files/02_largest_subgraph.html")

supporting_files/02_largest_subgraph.html


**修改结构**

In [128]:
# 获取所有的边
edges1 = list(sm_cluster_subgraph.edges)
edges1

[('cluster0', 'cluster2'),
 ('cluster0', 'cluster3'),
 ('cluster1', 'cluster0'),
 ('cluster1', 'cluster2'),
 ('cluster1', 'cluster3'),
 ('home_density_0', 'work_density_0'),
 ('home_density_0', 'married'),
 ('work_road_density_0', 'work_landusemix_0'),
 ('work_road_density_0', 'work_openratio_wide_0'),
 ('work_bus_0', 'work_landusemix_0'),
 ('work_bus_0', 'work_subway_0'),
 ('commute_distance_0', 'avg_daily_tube_freq0'),
 ('commute_distance_0', 'avg_daily_tube_freq1'),
 ('work_road_density_1', 'work_landusemix_1'),
 ('work_road_density_1', 'work_openratio_wide_1'),
 ('work_bus_1', 'work_landusemix_1'),
 ('work_bus_1', 'work_subway_1'),
 ('commute_distance_1', 'cluster2'),
 ('commute_distance_1', 'cluster3'),
 ('commute_distance_1', 'avg_daily_tube_freq3'),
 ('gender', 'cluster0'),
 ('gender', 'cluster1'),
 ('gender', 'cluster2'),
 ('gender', 'cluster3'),
 ('gender', 'age'),
 ('gender', 'income'),
 ('gender', 'state'),
 ('age', 'cluster0'),
 ('age', 'cluster1'),
 ('age', 'cluster2'),
 (

In [129]:
# 函数：提取节点名称中的数字部分
def get_time_index(node_name):
    match = re.search(r'(\d+)$', node_name)  # 匹配节点名中的数字部分
    return int(match.group(1)) if match else None

In [133]:
# 遍历所有边，删除不符合因果关系的边（时序性）
edges_to_remove = []
for parent, child in edges1:
    parent_time = get_time_index(parent)
    child_time = get_time_index(child)
    
    if parent_time is not None and child_time is not None and parent_time > child_time:  # 如果父节点时间序号大于子节点时间序号，删除这条边
        edges_to_remove.append((parent, child))

In [134]:
# 从图中删除这些边
sm_cluster_subgraph.remove_edges_from(edges_to_remove)

viz = plot_structure(
    sm_cluster_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.show("supporting_files/02_modified_structure.html")

supporting_files/02_modified_structure.html


In [137]:
edges2 = list(sm_cluster_subgraph.edges)

In [136]:
# 定义出行变量前缀
travel_cols = [
    'avg_daily_dist', 'avg_daily_dur', 'avg_daily_tube_dist', 'avg_daily_tube_freq',
    'avg_daily_slow_dist', 'avg_daily_slow_freq'
]

In [139]:
# 删除同一时期内出行变量之间的因果路径
edges_to_remove = []
for parent, child in edges2:
    # 检查父节点和子节点是否属于同一时期的变量
    if any(parent.startswith(prefix) for prefix in travel_cols) and any(child.startswith(prefix) for prefix in travel_cols):
        parent_time = get_time_index(parent)
        child_time = get_time_index(child)
        
        # 如果父节点和子节点属于同一时期，删除这条边
        if parent_time is not None and child_time is not None and parent_time == child_time:
            edges_to_remove.append((parent, child))

In [140]:
# 从图中删除这些边
sm_cluster_subgraph.remove_edges_from(edges_to_remove)

viz = plot_structure(
    sm_cluster_subgraph,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.show("supporting_files/02_modified_structure.html")

supporting_files/02_modified_structure.html


In [141]:
list(sm_cluster_subgraph.edges)

[('cluster0', 'cluster2'),
 ('cluster0', 'cluster3'),
 ('cluster1', 'cluster2'),
 ('cluster1', 'cluster3'),
 ('home_density_0', 'work_density_0'),
 ('home_density_0', 'married'),
 ('work_road_density_0', 'work_landusemix_0'),
 ('work_road_density_0', 'work_openratio_wide_0'),
 ('work_bus_0', 'work_landusemix_0'),
 ('work_bus_0', 'work_subway_0'),
 ('commute_distance_0', 'avg_daily_tube_freq0'),
 ('commute_distance_0', 'avg_daily_tube_freq1'),
 ('work_road_density_1', 'work_landusemix_1'),
 ('work_road_density_1', 'work_openratio_wide_1'),
 ('work_bus_1', 'work_landusemix_1'),
 ('work_bus_1', 'work_subway_1'),
 ('commute_distance_1', 'cluster2'),
 ('commute_distance_1', 'cluster3'),
 ('commute_distance_1', 'avg_daily_tube_freq3'),
 ('gender', 'cluster0'),
 ('gender', 'cluster1'),
 ('gender', 'cluster2'),
 ('gender', 'cluster3'),
 ('gender', 'age'),
 ('gender', 'income'),
 ('gender', 'state'),
 ('age', 'cluster0'),
 ('age', 'cluster1'),
 ('age', 'cluster2'),
 ('age', 'cluster3'),
 ('age'

In [11]:
# from IPython.display import display, HTML
# %matplotlib widget
# sm = StructureModel()
# sm.add_edges_from([
#     ('health', 'absences'),
#     ('health', 'G1')
# ])
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("supporting_files/01_simple_plot.html")
# # display(HTML("supporting_files/01_simple_plot.html"))

supporting_files/01_simple_plot.html
